In [19]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences #function used to make all text sequences the same length.
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout  #Converts word indexes into dense vector representations.

In [20]:

# Load Dataset (Auto Download)

vocab_size = 5000   #Keep only the top 5000 most frequent UNIQUE words from the dataset.
max_length = 200

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)
# from Dataset - Binary labels (0 = negative, 1 = positive)

In [21]:
# -----------------------------
# Padding Sequences
# -----------------------------
X_train = pad_sequences(X_train, maxlen=max_length)
X_test = pad_sequences(X_test, maxlen=max_length)

In [22]:
# -----------------------------
# Build LSTM Model
# -----------------------------
model = Sequential([
    Embedding(vocab_size, 128), # Embedding converts each number into a 128-dimensional vector.
    LSTM(128), # Output vector size = 128(neuron)
    Dropout(0.5),
    Dense(1, activation='sigmoid') #We have only 2 classes, need probability output, sigmoid is best for binary classification
    # Sigmoid gives output value as 0 or 1
])

#This prepares model for training.
model.compile(
    loss='binary_crossentropy', #Loss measures how wrong the model is. calculates error between actual and predict
    optimizer='adam',  # updates weights to reduce loss. Automatically adjusts weights
    metrics=['accuracy']
)



In [23]:
# -----------------------------
# Train Model
# -----------------------------
model.fit(
    X_train,
    y_train,
    epochs=5, #The model sees the entire training dataset once.
    batch_size=64, # Model processes 64 reviews at a time
    validation_data=(X_test, y_test)
)


Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 160s 406ms/step - accuracy: 0.7061 - loss: 0.5496 - val_accuracy: 0.8516 - val_loss: 0.3530
Epoch 2/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 149s 381ms/step - accuracy: 0.8802 - loss: 0.3029 - val_accuracy: 0.8738 - val_loss: 0.2970
Epoch 3/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 201s 378ms/step - accuracy: 0.9066 - loss: 0.2467 - val_accuracy: 0.8736 - val_loss: 0.3175
Epoch 4/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 147s 377ms/step - accuracy: 0.9156 - loss: 0.2111 - val_accuracy: 0.8718 - val_loss: 0.3218
Epoch 5/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 204s 383ms/step - accuracy: 0.9367 - loss: 0.1702 - val_accuracy: 0.8674 - val_loss: 0.3520


In [30]:
model.export("sentiment_model")
print("Model saved successfully.")
# Saving model and packages for deployment

Saved artifact at 'sentiment_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 200), dtype=tf.float32, name='keras_tensor_5')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  137924968806352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137925193409040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137925193404624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137925193405008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137925103502160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137925103503120: TensorSpec(shape=(), dtype=tf.resource, name=None)
Model saved successfully.


In [32]:
converter = tf.lite.TFLiteConverter.from_saved_model("sentiment_model")

# 🔧 LSTM Fix
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter._experimental_lower_tensor_list_ops = False

# ❌ No optimization
tflite_model = converter.convert()

with open("model_before_quant.tflite", "wb") as f:
    f.write(tflite_model)

print("Model before quantization saved.")

Model before quantization saved.


In [33]:
# convertin a model from Tensorflow to TFLite
# Convert to TFLite (FIXED FOR LSTM)
# TFLite  - lightweight version of: TensorFlow
# Load the exported SavedModel folder.
converter = tf.lite.TFLiteConverter.from_saved_model("sentiment_model")

# This tells the converter: “Apply default optimization techniques.”
# quantization happens here:
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# It tells TFLite: “Which operation types are allowed in the converted model?”
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,      # Standard TFLite ops
    tf.lite.OpsSet.SELECT_TF_OPS   # These allow TensorFlow operations that TFLite does not natively support.
]

# Disable tensor list lowering (required for LSTM)
converter._experimental_lower_tensor_list_ops = False
# Transforms your full TensorFlow model into a TensorFlow Lite model.
tflite_model = converter.convert() #It returns model as binary data.

with open("sentiment_model_optimized.tflite", "wb") as f:
    f.write(tflite_model)

print("Optimized TFLite model saved successfully.")   #This is the deployment file.

Optimized TFLite model saved successfully.


In [35]:
import os

before_size = os.path.getsize("model_before_quant.tflite")
after_size = os.path.getsize("sentiment_model_optimized.tflite")

print(f"Before Quantization: {before_size / 1024:.2f} KB")
print(f"After Quantization : {after_size / 1024:.2f} KB")

reduction = ((before_size - after_size) / before_size) * 100
print(f"Size Reduction: {reduction:.2f}%")

Before Quantization: 3026.31 KB
After Quantization : 779.58 KB
Size Reduction: 74.24%


In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.datasets import imdb

# -----------------------------
# Config (MUST match training)
# -----------------------------
vocab_size = 5000
max_length = 200

# Load word index
word_index = imdb.get_word_index()

# -----------------------------
# Load TFLite Model
# -----------------------------
interpreter = tf.lite.Interpreter(model_path="sentiment_model_optimized.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# -----------------------------
# Prediction Function
# -----------------------------
def predict_sentiment_production(text):
    words = text.lower().split()

    encoded = []
    for word in words:
        if word in word_index and word_index[word] < vocab_size:
            encoded.append(word_index[word] + 3)
        else:
            encoded.append(2)

    padded = pad_sequences([encoded], maxlen=max_length)
    padded = np.array(padded, dtype=np.float32)

    interpreter.set_tensor(input_details[0]['index'], padded)
    interpreter.invoke()

    prediction = interpreter.get_tensor(output_details[0]['index'])

    return "Positive 😊" if prediction[0][0] > 0.5 else "Negative 😞"

# -----------------------------
# Interactive Loop
# -----------------------------
while True:
    user_input = input("\nEnter a movie review (type 'exit' to quit): ")

    if user_input.lower() == "exit":
        print("Program ended.")
        break

    result = predict_sentiment_production(user_input)
    print("Predicted Sentiment:", result)


Enter a movie review (type 'exit' to quit): This movie delivers a powerful performance from the lead actor, and the storyline keeps you engaged from start to finish.
Predicted Sentiment: Positive 😊

Enter a movie review (type 'exit' to quit): I was genuinely impressed by the cinematography and the emotional depth of the characters throughout the film.
Predicted Sentiment: Positive 😊

Enter a movie review (type 'exit' to quit): An outstanding script combined with brilliant direction makes this one of the best films I’ve seen in years.
Predicted Sentiment: Positive 😊

Enter a movie review (type 'exit' to quit): Despite a slow beginning, the movie builds into a thrilling and satisfying conclusion that is well worth the wait.
Predicted Sentiment: Positive 😊

Enter a movie review (type 'exit' to quit): The plot was predictable and the acting felt forced, making it difficult to stay interested for more than a few minutes.
Predicted Sentiment: Negative 😞

Enter a movie review (type 'exit' to